In [1]:
#Import the libraries needed
import pandas as pd
from pathlib import Path

In [2]:
# Point to the folder that holds the CSV file
folder = Path("cat")

In [3]:
# Load every CSV in the folder into a dictionary
files = {f.stem: pd.read_csv(f) for f in folder.glob("*.csv")}

In [4]:
# Loop through each file to check its structure
for name, df in files.items():
    print(name, df.shape)
    print(list(df.columns))
    print()

companies_summary (2642, 4)
['country', 'field', 'disclosed_investment', 'estimated_investment']

companies_yearly_disclosed (14659, 5)
['country', 'field', 'year', 'disclosed_investment', 'complete']

companies_yearly_estimated (16587, 5)
['country', 'field', 'year', 'estimated_investment', 'complete']

patents_summary (2310, 4)
['country', 'field', 'num_patent_applications', 'num_patent_granted']

patents_yearly_applications (15972, 5)
['country', 'field', 'year', 'num_patent_applications', 'complete']

patents_yearly_granted (14223, 5)
['country', 'field', 'year', 'num_patent_granted', 'complete']

publications_summary (1053, 4)
['country', 'field', 'num_articles', 'num_citations']

publications_yearly_articles (8542, 5)
['country', 'field', 'year', 'num_articles', 'complete']

publications_yearly_citations (8303, 4)
['country', 'field', 'year', 'num_citations']



#### Step 2: Check the country spellings

In [5]:
# Collect every unique country name across all nine files
all_countries = set()
for df in files.values():
    all_countries.update(df["country"].dropna().unique())

In [6]:
# Show how many unique country names there are in total
print(len(all_countries))

205


In [7]:
# Search for possible spellings of each of our six countries
# We search for partial words so we catch variants like "USA" or "United States of America"
keywords = ["swed", "united", "usa", "america", "german", "south africa", "china", "india"]
for word in keywords:
    matches = sorted([c for c in all_countries if word in str(c).lower()])
    print(word, "->", matches)

swed -> ['Sweden']
united -> ['United Arab Emirates', 'United Kingdom', 'United States']
usa -> []
america -> ['North America and the Caribbean', 'South and Central America']
german -> ['Germany']
south africa -> ['South Africa']
china -> ['China']
india -> ['India']


#### Step 3: Filter to the six countries

In [8]:
# Define the six countries, grouped as developed and developing
countries = {
    "developed": ["Sweden", "United States", "Germany"],
    "developing": ["South Africa", "China", "India"],
}

In [9]:
# Combine both groups into one list for filtering
selected = countries["developed"] + countries["developing"]


In [10]:
# Create a dictionary to hold the filtered DataFrames
filtered = {}

for name, df in files.items():
    # Keep only the rows where the country is one of our six
    filtered[name] = df[df["country"].isin(selected)].copy()

In [11]:
# Map each country to its group label
group_map = {c: "Developed" for c in countries["developed"]}
group_map.update({c: "Developing" for c in countries["developing"]})

In [12]:
# Add a "group" column to every filtered DataFrame
for df in filtered.values():
    df["group"] = df["country"].map(group_map)

In [13]:
# Check each file: rows kept, number of countries, and which ones are present
for name, df in filtered.items():
    print(name, len(df), "rows,", df["country"].nunique(), "countries")
    print(sorted(df["country"].unique()))
    print()

companies_summary 215 rows, 6 countries
['China', 'Germany', 'India', 'South Africa', 'Sweden', 'United States']

companies_yearly_disclosed 1960 rows, 6 countries
['China', 'Germany', 'India', 'South Africa', 'Sweden', 'United States']

companies_yearly_estimated 2055 rows, 6 countries
['China', 'Germany', 'India', 'South Africa', 'Sweden', 'United States']

patents_summary 198 rows, 6 countries
['China', 'Germany', 'India', 'South Africa', 'Sweden', 'United States']

patents_yearly_applications 1980 rows, 6 countries
['China', 'Germany', 'India', 'South Africa', 'Sweden', 'United States']

patents_yearly_granted 1815 rows, 6 countries
['China', 'Germany', 'India', 'South Africa', 'Sweden', 'United States']

publications_summary 36 rows, 6 countries
['China', 'Germany', 'India', 'South Africa', 'Sweden', 'United States']

publications_yearly_articles 388 rows, 6 countries
['China', 'Germany', 'India', 'South Africa', 'Sweden', 'United States']

publications_yearly_citations 387 rows, 

#### Step 5: Save the filtered files

In [14]:
# Create a separate folder so the raw files stay untouched
Path("clean").mkdir(exist_ok=True)

In [15]:
# Save each filtered DataFrame as a new CSV
for name, df in filtered.items():
    df.to_csv(f"clean/{name}_6countries.csv", index=False)


In [16]:
# Confirm the files were created
print(list(Path("clean").glob("*.csv")))

[WindowsPath('clean/companies_summary_6countries.csv'), WindowsPath('clean/companies_yearly_disclosed_6countries.csv'), WindowsPath('clean/companies_yearly_estimated_6countries.csv'), WindowsPath('clean/patents_summary_6countries.csv'), WindowsPath('clean/patents_yearly_applications_6countries.csv'), WindowsPath('clean/patents_yearly_granted_6countries.csv'), WindowsPath('clean/publications_summary_6countries.csv'), WindowsPath('clean/publications_yearly_articles_6countries.csv'), WindowsPath('clean/publications_yearly_citations_6countries.csv')]


#### Step 6: First EDA checks on every file

In [17]:
# Build one summary row per file
rows = []
for name, df in filtered.items():
    rows.append({
        "file": name,
        "rows": len(df),
        # Total number of empty cells in the file
        "missing_cells": int(df.isnull().sum().sum()),
        # Number of fully duplicated rows
        "duplicates": int(df.duplicated().sum()),
        # Number of distinct fields covered
        "fields": df["field"].nunique(),
        # Year range, only for the yearly files
        "first_year": df["year"].min() if "year" in df.columns else None,
        "last_year": df["year"].max() if "year" in df.columns else None,
    })

# Show the summary as a single table
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

# List only the columns that actually have missing values
for name, df in filtered.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(name, dict(missing))

                         file  rows  missing_cells  duplicates  fields  first_year  last_year
            companies_summary   215              0           0      37         NaN        NaN
   companies_yearly_disclosed  1960              0           0      37      2016.0     2026.0
   companies_yearly_estimated  2055              0           0      37      2016.0     2026.0
              patents_summary   198              0           0      33         NaN        NaN
  patents_yearly_applications  1980              0           0      33      2016.0     2026.0
       patents_yearly_granted  1815              0           0      33      2016.0     2026.0
         publications_summary    36              0           0       6         NaN        NaN
 publications_yearly_articles   388              0           0       6      2016.0     2026.0
publications_yearly_citations   387              0           0       6      2016.0     2026.0


#### Step 7: Understand the complete column

In [18]:
# Check what values the "complete" column holds and how often
print(filtered["patents_yearly_applications"]["complete"].value_counts())

# See how "complete" varies by year, since it often flags the latest years as unfinished
print(
    filtered["patents_yearly_applications"]
    .groupby("year")["complete"]
    .value_counts()
    .unstack()
    .tail(10)
)

complete
True     1188
False     792
Name: count, dtype: int64
complete  False  True 
year                  
2017        NaN  198.0
2018        NaN  198.0
2019        NaN  198.0
2020        NaN  198.0
2021        NaN  198.0
2022      198.0    NaN
2023      198.0    NaN
2024      198.0    NaN
2025      165.0    NaN
2026       33.0    NaN


#### Step 8: Check the flag in every yearly file

In [19]:
# The yearly files that have a "complete" column
yearly_files = [
    "companies_yearly_disclosed",
    "companies_yearly_estimated",
    "patents_yearly_applications",
    "patents_yearly_granted",
    "publications_yearly_articles",
]

for name in yearly_files:
    df = filtered[name]
    # Find the earliest and latest year in the file
    print(name, "covers", df["year"].min(), "to", df["year"].max())
    # Find the last year flagged as complete
    print("  Last complete year:", df[df["complete"] == True]["year"].max())
    # Find the first year flagged as incomplete
    print("  First incomplete year:", df[df["complete"] == False]["year"].min())

companies_yearly_disclosed covers 2016 to 2026
  Last complete year: 2025
  First incomplete year: 2026
companies_yearly_estimated covers 2016 to 2026
  Last complete year: 2025
  First incomplete year: 2026
patents_yearly_applications covers 2016 to 2026
  Last complete year: 2021
  First incomplete year: 2022
patents_yearly_granted covers 2016 to 2026
  Last complete year: 2019
  First incomplete year: 2020
publications_yearly_articles covers 2016 to 2026
  Last complete year: 2024
  First incomplete year: 2025


#### Step 9: Check citations and the common window

In [20]:
# The citations file has no "complete" column, so check its years directly
cit = filtered["publications_yearly_citations"]
print("Citations years:", cit["year"].min(), "to", cit["year"].max())

# Count rows per year to spot years that look partial
print(cit.groupby("year").size())

# Last complete year for each flagged file, stored in a dictionary
last_complete = {}
for name in yearly_files:
    df = filtered[name]
    last_complete[name] = df[df["complete"] == True]["year"].max()

# The common window ends at the earliest of these cut-off years
print("Common window ends at:", min(last_complete.values()))

# The window if patents granted is excluded
without_granted = {k: v for k, v in last_complete.items() if k != "patents_yearly_granted"}
print("Window without patents granted ends at:", min(without_granted.values()))

Citations years: 2016 to 2026
year
2016    35
2017    36
2018    36
2019    36
2020    36
2021    36
2022    36
2023    36
2024    36
2025    36
2026    28
dtype: int64
Common window ends at: 2019
Window without patents granted ends at: 2021


#### Step 10: Compare patent counts by year within 2018 to 2024

In [21]:
# Patent applications: total per year inside the 2018 to 2024 window
apps = filtered["patents_yearly_applications"]
apps = apps[(apps["year"] >= 2018) & (apps["year"] <= 2024)]
print(apps.groupby("year")["num_patent_applications"].sum())

# Patents granted: total per year inside the same window
granted = filtered["patents_yearly_granted"]
granted = granted[(granted["year"] >= 2018) & (granted["year"] <= 2024)]
print(granted.groupby("year")["num_patent_granted"].sum())

year
2018     260574
2019     371403
2020     519120
2021     627160
2022     689047
2023     838700
2024    1079338
Name: num_patent_applications, dtype: int64
year
2018    183075
2019    262804
2020    364391
2021    408706
2022    375130
2023    312929
2024    351690
Name: num_patent_granted, dtype: int64


#### Step 11: Read the readme

In [22]:
# Open the readme file from the main project folder and print its contents
with open("readme.md", "r", encoding="utf-8") as f:
    print(f.read())

# Scholarly publications on artificial intelligence per million people - Data package

This data package contains the data that powers the chart ["Scholarly publications on artificial intelligence per million people"](https://ourworldindata.org/grapher/scholarly-publications-on-artificial-intelligence-per-million-people?tab=line&country=ZAF~USA~DEU~CHN~IND~SWE&mapSelect=ZAF~USA~DEU~CHN~IND~SWE&v=1&csvType=filtered&useColumnShortNames=false) on the Our World in Data website. It was downloaded on September 29, 2026.

### Active Filters

A filtered subset of the full data was downloaded. The following filters were applied:
- country: ZAF, USA, DEU, CHN, IND, SWE
- tab: line
- mapSelect: ZAF~USA~DEU~CHN~IND~SWE

## CSV structure

Each row is an observation for an entity (usually a country or region) at a timepoint.

- "Entity" — the name of the entity, e.g. "United States".
- "Code" — our internal entity code. For most countries this is the [ISO alpha-3](https://en.wikipedia.org/wiki/ISO_3

#### Step 12: Build the country and year table

In [23]:
# Import reduce so we can merge several tables in one go
from functools import reduce

# The analysis window your team agreed on
START, END = 2018, 2024

# The yearly file and the value column to use from each one
value_cols = {
    "companies_yearly_disclosed": "disclosed_investment",
    "companies_yearly_estimated": "estimated_investment",
    "patents_yearly_applications": "num_patent_applications",
    "patents_yearly_granted": "num_patent_granted",
    "publications_yearly_articles": "num_articles",
    "publications_yearly_citations": "num_citations",
}

parts = []
for name, col in value_cols.items():
    df = filtered[name]
    # Keep only the "All" rows, which already hold the true total for each country and year
    df = df[df["field"] == "All"]
    # Keep only the years inside the window
    df = df[(df["year"] >= START) & (df["year"] <= END)]
    # Keep just the columns we need
    parts.append(df[["country", "year", col]])

# Merge the six tables on country and year
panel = reduce(lambda left, right: pd.merge(left, right, on=["country", "year"], how="outer"), parts)

# Add the developed or developing label
panel["group"] = panel["country"].map(group_map)

# Check the result, we expect 42 rows (6 countries x 7 years)
print(panel.shape)
print(panel.isnull().sum())
panel.head(10)

(42, 9)
country                    0
year                       0
disclosed_investment       0
estimated_investment       0
num_patent_applications    0
num_patent_granted         4
num_articles               0
num_citations              0
group                      0
dtype: int64


,country,year,disclosed_investment,estimated_investment,num_patent_applications,num_patent_granted,num_articles,num_citations,group
0,China,2018,9394,11141,30020,17896.0,36754,1551273,Developing
1,China,2019,5551,7338,48119,29712.0,48935,2022008,Developing
2,China,2020,12486,15084,71883,46843.0,56769,2157692,Developing
3,China,2021,11569,17750,92950,57876.0,72439,2104518,Developing
4,China,2022,8579,15600,107425,56719.0,82800,1725881,Developing
5,China,2023,5714,11787,135288,48826.0,104292,1633678,Developing
6,China,2024,9637,16224,176802,57064.0,110110,913729,Developing
7,Germany,2018,585,3923,736,503.0,7256,340082,Developed
8,Germany,2019,948,1507,987,608.0,8550,370698,Developed
9,Germany,2020,1169,4359,1186,647.0,10050,428839,Developed


In [24]:
# Show only the rows where granted patents are missing
missing_granted = panel[panel["num_patent_granted"].isnull()]
print(missing_granted[["country", "year", "num_patent_granted"]])

   country  year  num_patent_granted
17   India  2021                 NaN
18   India  2022                 NaN
19   India  2023                 NaN
20   India  2024                 NaN


#### Step 14: Calculate the statistics for each country

In [25]:
# The six indicator columns we want statistics for
indicators = [
    "disclosed_investment",
    "estimated_investment",
    "num_patent_applications",
    "num_patent_granted",
    "num_articles",
    "num_citations",
]

# Helper functions for the quartiles, since pandas has no built-in IQR
def q1(x):
    return x.quantile(0.25)

def q3(x):
    return x.quantile(0.75)

def iqr(x):
    return x.quantile(0.75) - x.quantile(0.25)

# Calculate the statistics for every indicator, grouped by country
stats = panel.groupby("country")[indicators].agg(
    ["count", "mean", "median", "std", "min", "max", q1, q3, iqr]
)

# Rename the helper function columns so they read nicely
stats = stats.rename(columns={"q1": "Q1", "q3": "Q3", "iqr": "IQR"})

# Show the statistics for one indicator at a time to keep the output readable
stats["num_patent_applications"].round(1)

,count,mean,median,std,min,max,Q1,Q3,IQR
country,,,,,,,,,
China,7,94641.0,92950.0,50706.7,30020,176802,60001.0,121356.5,61355.5
Germany,7,1133.9,1174.0,260.4,736,1503,979.5,1282.5,303.0
India,7,45.6,11.0,99.0,2,270,6.0,12.0,6.0
South Africa,7,49.3,23.0,52.3,5,134,6.5,85.0,78.5
Sweden,7,28.6,31.0,6.6,20,36,23.0,33.5,10.5
United States,7,12737.7,14638.0,4502.0,5277,16913,10096.0,16072.0,5976.0


#### Step 15: Look at the yearly values for the three low countries

In [30]:
# Show the patent applications per year for India, Sweden and South Africa
low = panel[panel["country"].isin(["India", "Sweden", "South Africa"])]
print(low[["country", "year", "num_patent_applications"]].to_string(index=False))

     country  year  num_patent_applications
       India  2018                      270
       India  2019                       12
       India  2020                       12
       India  2021                        6
       India  2022                        6
       India  2023                       11
       India  2024                        2
South Africa  2018                        7
South Africa  2019                        5
South Africa  2020                        6
South Africa  2021                       23
South Africa  2022                       71
South Africa  2023                       99
South Africa  2024                      134
      Sweden  2018                       33
      Sweden  2019                       20
      Sweden  2020                       31
      Sweden  2021                       26
      Sweden  2022                       36
      Sweden  2023                       34
      Sweden  2024                       20


In [31]:
# Show the statistics for granted patents
stats["num_patent_granted"].round(1)

,count,mean,median,std,min,max,Q1,Q3,IQR
country,,,,,,,,,
China,7,44990.9,48826.0,15465.3,17896.0,57876.0,38277.5,56891.5,18614.0
Germany,7,462.1,497.0,154.0,220.0,647.0,380.0,555.5,175.5
India,3,71.7,8.0,113.8,4.0,203.0,6.0,105.5,99.5
South Africa,7,48.1,22.0,52.6,3.0,132.0,5.0,85.0,80.0
Sweden,7,17.9,21.0,6.2,9.0,24.0,14.0,21.5,7.5
United States,7,9820.6,12264.0,5346.9,1531.0,15199.0,6082.0,13793.0,7711.0


In [32]:
# Show the statistics for disclosed investment
stats["disclosed_investment"].round(1)

,count,mean,median,std,min,max,Q1,Q3,IQR
country,,,,,,,,,
China,7,8990.0,9394.0,2651.5,5551,12486,7146.5,10603.0,3456.5
Germany,7,2204.6,2415.0,1344.0,585,4007,1058.5,3154.0,2095.5
India,7,4700.3,1902.0,7094.0,503,20513,1249.5,3742.5,2493.0
South Africa,7,72.3,59.0,54.1,9,141,33.0,115.5,82.5
Sweden,7,2149.6,601.0,2445.9,68,5449,165.5,4299.0,4133.5
United States,7,69025.3,59585.0,36691.3,35690,128834,40038.5,89495.5,49457.0


In [33]:
# Show the statistics for estimated investment
stats["estimated_investment"].round(1)

,count,mean,median,std,min,max,Q1,Q3,IQR
country,,,,,,,,,
China,7,13560.6,15084.0,3625.0,7338,17750,11464.0,15912.0,4448.0
Germany,7,4874.7,5160.0,1871.1,1507,7394,4141.0,5890.0,1749.0
India,7,6450.6,2541.0,8637.6,892,25214,1898.0,6355.5,4457.5
South Africa,7,150.1,150.0,61.3,70,242,111.5,183.0,71.5
Sweden,7,4597.6,3600.0,6450.3,130,18369,235.0,4807.0,4572.0
United States,7,115017.6,103086.0,48130.8,57431,184951,87143.0,142684.5,55541.5


In [34]:
# Show the statistics for publication articles
stats["num_articles"].round(1)

,count,mean,median,std,min,max,Q1,Q3,IQR
country,,,,,,,,,
China,7,73157.0,72439.0,27715.4,36754,110110,52852.0,93546.0,40694.0
Germany,7,10875.1,11870.0,2353.3,7256,13936,9300.0,12232.0,2932.0
India,7,29450.4,29077.0,13930.7,14100,51154,18459.5,37451.5,18992.0
South Africa,7,769.9,804.0,280.1,383,1113,580.5,964.0,383.5
Sweden,7,2005.0,2115.0,520.9,1208,2650,1672.0,2359.0,687.0
United States,7,42598.7,43440.0,7375.1,29526,50230,39659.0,47838.5,8179.5


In [35]:
# Show the statistics for citations
stats["num_citations"].round(1)

,count,mean,median,std,min,max,Q1,Q3,IQR
country,,,,,,,,,
China,7,1729825.6,1725881.0,431275.1,913729,2157692,1592475.5,2063263.0,470787.5
Germany,7,310380.9,340082.0,100807.8,123615,428839,269147.0,370918.0,101771.0
India,7,348268.9,372532.0,78205.5,242043,438782,286079.5,406183.0,120103.5
South Africa,7,18100.1,18366.0,4069.4,10025,21838,17164.5,21071.5,3907.0
Sweden,7,59366.7,57608.0,16583.3,28688,77885,54056.0,71637.0,17581.0
United States,7,1623784.9,1755131.0,694187.4,563732,2313570,1122827.5,2244203.0,1121375.5


#### Step 16: Look at the field names

In [27]:
# List the distinct fields in the patent applications file
fields = sorted(filtered["patents_yearly_applications"]["field"].unique())
print(len(fields))
print(fields)

33
['Agriculture', 'All', 'Analytics and Algorithms', 'Banking and Finance', 'Bio-Inspired Computing', 'Business', 'Computer Vision', 'Computing in Government', 'Control Methods', 'Distributed AI', 'Education', 'Energy Management', 'Fuzzy Logic', 'Industry and Manufacturing', 'Knowledge Representation', 'Life Sciences', 'Logic-Based Knowledge Systems', 'Markov and Bayesian Models', 'Measuring and Testing', 'Natural Language Processing', 'Neural Networks', 'Ontology Engineering', 'Other and Unspecified', 'Personal Devices and Computing', 'Physical Sciences and Engineering', 'Planning and Scheduling', 'Robotics', 'Search Methods', 'Security', 'Spatial and Physical Reasoning', 'Speech Processing', 'Telecommunications', 'Transportation']


#### Step 17: Check the A11 field

In [28]:
# Take China's patent applications in 2021 from the filtered file
check = filtered["patents_yearly_applications"]
check = check[(check["country"] == "China") & (check["year"] == 2021)]

# The value in the "All" row
all_value = check.loc[check["field"] == "All", "num_patent_applications"].iloc[0]

# The sum of every other field
other_sum = check.loc[check["field"] != "All", "num_patent_applications"].sum()

print("All row:", all_value)
print("Sum of other fields:", other_sum)

# Check which files have an "All" field
for name, df in filtered.items():
    print(name, "has All:", "All" in df["field"].unique())

All row: 92950
Sum of other fields: 434078
companies_summary has All: True
companies_yearly_disclosed has All: True
companies_yearly_estimated has All: True
patents_summary has All: True
patents_yearly_applications has All: True
patents_yearly_granted has All: True
publications_summary has All: True
publications_yearly_articles has All: True
publications_yearly_citations has All: True


#### Step 18: Build the final summary tabl

In [36]:
# Make sure the output folder exists
Path("clean").mkdir(exist_ok=True)

# Collect the key statistics for every indicator into one table
rows = []
for col in indicators:
    s = stats[col]
    for country in s.index:
        rows.append({
            "indicator": col,
            "country": country,
            # Number of years the statistics are based on
            "count": s.loc[country, "count"],
            "mean": s.loc[country, "mean"],
            "median": s.loc[country, "median"],
            "std": s.loc[country, "std"],
            "min": s.loc[country, "min"],
            "max": s.loc[country, "max"],
            # Range is the largest value minus the smallest
            "range": s.loc[country, "max"] - s.loc[country, "min"],
            "Q1": s.loc[country, "Q1"],
            "Q3": s.loc[country, "Q3"],
            "IQR": s.loc[country, "IQR"],
            # Coefficient of variation lets you compare spread across countries of different size
            "CV": s.loc[country, "std"] / s.loc[country, "mean"],
        })

# Turn the list into a table and round it
summary = pd.DataFrame(rows).round(2)

# Save it as a CSV that opens in Excel
summary.to_csv("clean/summary_statistics.csv", index=False)

# Show the first rows as a check
summary.head(12)

,indicator,country,count,mean,median,std,min,max,range,Q1,Q3,IQR,CV
0,disclosed_investment,China,7,8990.00,9394.0,2651.51,5551.0,12486.0,6935.0,7146.5,10603.0,3456.5,0.29
1,disclosed_investment,Germany,7,2204.57,2415.0,1344.05,585.0,4007.0,3422.0,1058.5,3154.0,2095.5,0.61
2,disclosed_investment,India,7,4700.29,1902.0,7094.04,503.0,20513.0,20010.0,1249.5,3742.5,2493.0,1.51
3,disclosed_investment,South Africa,7,72.29,59.0,54.12,9.0,141.0,132.0,33.0,115.5,82.5,0.75
4,disclosed_investment,Sweden,7,2149.57,601.0,2445.86,68.0,5449.0,5381.0,165.5,4299.0,4133.5,1.14
5,disclosed_investment,United States,7,69025.29,59585.0,36691.33,35690.0,128834.0,93144.0,40038.5,89495.5,49457.0,0.53
6,estimated_investment,China,7,13560.57,15084.0,3625.04,7338.0,17750.0,10412.0,11464.0,15912.0,4448.0,0.27
7,estimated_investment,Germany,7,4874.71,5160.0,1871.09,1507.0,7394.0,5887.0,4141.0,5890.0,1749.0,0.38
8,estimated_investment,India,7,6450.57,2541.0,8637.57,892.0,25214.0,24322.0,1898.0,6355.5,4457.5,1.34
9,estimated_investment,South Africa,7,150.14,150.0,61.33,70.0,242.0,172.0,111.5,183.0,71.5,0.41


In [37]:
# Check the table has 36 rows (6 indicators x 6 countries)
print(summary.shape)

# Show only the rows based on fewer than 7 years
print(summary[summary["count"] < 7][["indicator", "country", "count"]])

(36, 13)
             indicator country  count
20  num_patent_granted   India      3
